# COMPLETENESS (v3)

v3: uses `library.v3` (see `library/v3/README.md`) and the frozen v2 polluters (`library.v2.my_functions`).
Every experiment is run with `run_experiment` over seeds `[0, 1, 2]`; the figures come from the saved results JSON
(`Plot/v3/results/<exp_id>.json`, figures in `Plot/v3/<exp_id>/`). The main metric is the test nRMSE
(RMSE / std of the clean target), paired across conditions and models (same data, same missingness mask, same folds).

Set `QUICK = True` (or the environment variable `DIQ_QUICK=1`) for a smoke run: x axes are cut to 2 points and one seed.

Imputation conditions: `mean` is the default baseline; `std_constant` is a naive constant (std), not a recognised
method (it reproduces the original study); `most_frequent` is degenerate on continuous data; `native` is
`HGBRegressor` on raw NaN (a reference model, not one of the six study models); `clean` is the unpolluted reference.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.datasets import make_regression

from library.v2 import my_functions as mf
from library.v3 import evaluation as ev
from library.v3.data import make_regression_data
from library.v3.runner import run_experiment, load_results, summarise
from library.v3.plots import plot_experiment

SEEDS = [0, 1, 2]
N_JOBS = 8
QUICK = False
if os.environ.get("DIQ_QUICK") == "1":
    QUICK = True
if QUICK:
    SEEDS = [0]

MODELS = ev.STUDY_MODELS
MODELS_NATIVE = ev.STUDY_MODELS + ["HGBRegressor"]  # HGBRegressor is only needed for the 'native' reference
N_SAMPLES = 1000
N_AXIS = [100 * k for k in range(1, 11)]  # sample-size axis, 100..1000

warnings.simplefilter("ignore", FutureWarning)


def axis(values):
    values = list(values)
    return values[:2] if QUICK else values


def show(exp_id):
    """Figures from the results JSON, then the compact test-nRMSE pivot (mean over seeds; rows x, columns condition/model)."""
    plot_experiment(exp_id)
    s = summarise(load_results(exp_id))
    display(s.pivot_table(index="x", columns=["condition", "model"], values="mean", sort=False).round(3))
    return s


def meta_level(x, **extra):
    return {"requested": x, **extra}


# ---- data generators (top level, picklable) ----
def make_data_n(seed, x, *, n_features=3, n_informative=3):
    return make_regression_data(int(x), n_features, n_informative, seed)


def make_data_3(seed):
    return make_regression_data(N_SAMPLES, 3, 3, seed)


def make_data_9_4(seed):
    return make_regression_data(N_SAMPLES, 9, 4, seed)


def make_data_10_10(seed):
    return make_regression_data(N_SAMPLES, 10, 10, seed)


def make_data_5_3_n(seed, x):
    return make_regression_data(int(x), 5, 3, seed)


def make_data_ranked(seed, *, n_features=10, n_informative=5, noise_ratio=0.2):
    """Same protocol as make_regression_data, but columns are ordered by true |coef| (descending) so that
    column k is the k-th most important feature; columns 0..n_informative-1 are the informative ones."""
    X, signal, coef = make_regression(n_samples=N_SAMPLES, n_features=n_features, n_informative=n_informative,
                                      noise=0.0, shuffle=False, coef=True, random_state=seed)
    order = np.argsort(-np.abs(coef), kind="stable")
    rng = np.random.default_rng(seed)
    y = signal + rng.normal(0.0, noise_ratio * np.std(signal), size=signal.shape)
    return pd.DataFrame(X[:, order]), y, float(np.std(y))


# ---- polluters: (X, y, x, seed) -> (X, y, meta) ----
def pollute_mcar(X, y, x, seed):
    return mf.pollute_data_mcar(X, x, seed), y, meta_level(x, mechanism="MCAR", features="all")


def pollute_mcar_one_ranked(X, y, x, seed):
    X = mf.pollute_data_mcar_for_one_feature(X, int(x), 0.2, seed)
    return X, y, meta_level(0.2, mechanism="MCAR", feature_rank=int(x), informative=bool(int(x) < 5))


def pollute_incremental(X, y, x, seed):
    return (mf.pollute_data_mcar_for_each_feature_in_incremental_way(X, seed), y,
            meta_level("column i -> 10*i %", mechanism="MCAR incremental", n_samples=int(x)))


def pollute_mar(X, y, x, seed):
    X = mf.pollute_data_mar(X, 1, 2, seed, q=1 - x)
    return X, y, meta_level(x, mechanism="MAR", driver_feature=1, dependent_feature=2)


def pollute_mnar_upper(X, y, x, seed):
    X = mf.pollute_data_mnar(X, 1, seed, q_low=1 - x, q_high=1.0)
    return X, y, meta_level(x, mechanism="MNAR upper tail", feature=1)


def pollute_most(X, y, x, seed):
    return mf.pollute_most_important_features(X, y, x, seed), y, meta_level(x, mechanism="MCAR", features="3 most important")


def pollute_less(X, y, x, seed):
    return mf.pollute_less_important_features(X, y, x, seed), y, meta_level(x, mechanism="MCAR", features="3 least important")


def pollute_mixed(X, y, x, seed):
    # Both feature sets are ranked on the clean data (each v2 call gets the unpolluted X), then the two NaN patterns are merged.
    least = mf.pollute_less_important_features(X, y, x, seed)
    most = mf.pollute_most_important_features(X, y, x, seed)
    cols_least = [j for j in range(X.shape[1]) if least.iloc[:, j].isna().any() and not X.iloc[:, j].isna().any()]
    cols_most = [j for j in range(X.shape[1]) if most.iloc[:, j].isna().any() and not X.iloc[:, j].isna().any()]
    assert len(cols_least) == 3 and len(cols_most) == 3 and not set(cols_least) & set(cols_most), "least/most sets must be 3 + 3 disjoint columns"
    out = least.copy()
    for j in cols_most:
        out.isetitem(j, most.iloc[:, j].to_numpy())
    return out, y, meta_level(x, mechanism="MCAR", features="3 least + 3 most important (ranked on clean data)",
                              least_columns=cols_least, most_columns=cols_most)


def pollute_mcar_mnar(X, y, x, seed):
    # MNAR first, on the complete column (exactly the upper 25%), then MCAR on all features; the union rate on feature 0 is about 0.25 + 0.75 * x.
    X = mf.pollute_data_mnar(X, 0, seed, q_low=0.75, q_high=1.0)
    X = mf.pollute_data_mcar(X, x, seed)
    return X, y, meta_level(x, mechanism="MNAR upper tail 25% on feature 0, then MCAR on all", mnar_feature=0, requested_mnar_rate=0.25,
                            requested_mcar_rate=x, realised_missing_rate=[float(r) for r in X.isna().mean()])


def pollute_none(X, y, x, seed):
    return X, y, meta_level(0.0, mechanism="none")


## Experiment 0. Baseline with complete data

**Measures:** test nRMSE of the six study models on complete data as the sample size grows (n = 100..1000, step 100). **Defect:** none. **Conditions:** `clean` only. The data are regenerated for every n (`make_data(seed, x)`), so the curve shows the learning-curve effect that later sample-size experiments are compared with.

In [ ]:
run_experiment("c00_baseline", "Completeness - Baseline with complete data", "Number of samples", axis(N_AXIS),
               make_data_n, pollute_none, models=MODELS, conditions=["clean"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c00 = show("c00_baseline")

## Experiment 1. MCAR with low percentages

**Measures:** test nRMSE when a share of cells (2.5%..25%) of all 3 features (1000 rows, all informative) is missing. **Defect:** MCAR, each cell removed independently with the requested probability. **Conditions:** `clean` reference; imputers `mean` (the default baseline), `median`, `most_frequent` (degenerate on continuous data), `std_constant` (naive constant, std; not a recognised method), `knn`, `iterative`, `mean_indicator`; `native` (HGBRegressor on raw NaN) as a separate reference.

In [ ]:
run_experiment("c01_mcar_low", "Completeness - MCAR with low percentages", "Percentage of missing values", axis([round(0.025 * k, 3) for k in range(1, 11)]),
               make_data_3, pollute_mcar, models=MODELS_NATIVE, conditions=["clean", "mean", "median", "most_frequent", "std_constant", "knn", "iterative", "mean_indicator", "native"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c01 = show("c01_mcar_low")

## Experiment 2. MCAR with high percentages

**Measures:** as C1 for 27.5%..50% missing cells. **Defect:** MCAR on all 3 features. **Conditions:** `clean`, `mean`, `median`, `knn`, `iterative`, `mean_indicator`, plus `native` (HGBRegressor) as a reference. `most_frequent` and `std_constant` are left out here (degenerate / naive, shown in C1).

In [ ]:
run_experiment("c02_mcar_high", "Completeness - MCAR with high percentages", "Percentage of missing values", axis([round(0.025 * k, 3) for k in range(11, 21)]),
               make_data_3, pollute_mcar, models=MODELS_NATIVE, conditions=["clean", "mean", "median", "knn", "iterative", "mean_indicator", "native"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c02 = show("c02_mcar_high")

## Experiment 3. Fixed MCAR on a single feature of different importance

**Measures:** test nRMSE when 20% of one feature is MCAR-missing, as a function of the feature's importance. **Defect:** MCAR 20% on one column; 10 features, 5 informative. Columns are ordered by true |coefficient| (descending, `make_regression(shuffle=False, coef=True)`), so x = feature rank 0..9 is interpretable: ranks 0..4 are informative, ranks 5..9 are noise (the `informative` flag is stored in `meta`). **Conditions:** `clean`, `mean`, `median`, `iterative`, plus `native` reference.

In [ ]:
run_experiment("c03_mcar_one_feature", "Completeness - fixed MCAR 20% on one feature, by feature rank", "Feature rank by true |coef| (0 = most important)", axis(list(range(10))),
               make_data_ranked, pollute_mcar_one_ranked, models=MODELS_NATIVE, conditions=["clean", "mean", "median", "iterative", "native"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c03 = show("c03_mcar_one_feature")

## Experiment 4. Incremental MCAR per feature with growing sample size

**Measures:** test nRMSE as n grows (100..1000) while column i has 10*i % missing values (0%, 10%, ..., 40%). **Defect:** MCAR with per-column rates; 5 features, 3 informative. The data are regenerated for every n (`make_data(seed, x)`). **Note:** the v2 polluter uses the same seed for every column, so the missing-row sets are nested across columns (column i is a subset of column i+1); missingness is correlated across columns (v2 behaviour, kept). **Conditions:** `clean`, `mean`, `median`, `iterative`, plus `native` reference.

In [ ]:
run_experiment("c04_incremental", "Completeness - incremental MCAR per feature, growing dataset", "Number of samples", axis(N_AXIS),
               make_data_5_3_n, pollute_incremental, models=MODELS_NATIVE, conditions=["clean", "mean", "median", "iterative", "native"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c04 = show("c04_incremental")

## Experiment 5. MAR (missing at random)

**Measures:** test nRMSE for a missing rate of 5%..50% in feature 2. **Defect:** MAR, feature 2 is removed where feature 1 is above its (1 - x) quantile (v2 `pollute_data_mar(q=1-x)`), so the realised rate is x and depends on an observed feature. 3 informative features, 1000 rows. **Conditions:** `clean`, `mean`, `median`, `iterative`, `mean_indicator`, plus `native` reference.

In [ ]:
run_experiment("c05_mar", "Completeness - MAR", "Percentage of missing values in feature 2", axis([round(0.05 * k, 2) for k in range(1, 11)]),
               make_data_3, pollute_mar, models=MODELS_NATIVE, conditions=["clean", "mean", "median", "iterative", "mean_indicator", "native"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c05 = show("c05_mar")

## Experiment 6. MNAR upper tail (self-masking)

**Measures:** test nRMSE for a missing rate of 5%..50% in feature 1. **Defect:** MNAR, feature 1 is removed when its own value exceeds its (1 - x) quantile (v2 `pollute_data_mnar(q_low=1-x, q_high=1.0)`), so the largest values vanish. 3 informative features, 1000 rows. **Conditions:** `clean`, `mean`, `iterative`, `mean_indicator`, plus `native` reference.

In [ ]:
run_experiment("c06_mnar", "Completeness - MNAR upper tail", "Percentage of missing values in feature 1", axis([round(0.05 * k, 2) for k in range(1, 11)]),
               make_data_3, pollute_mnar_upper, models=MODELS_NATIVE, conditions=["clean", "mean", "iterative", "mean_indicator", "native"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c06 = show("c06_mnar")

## Experiment 7. MCAR in the most informative features

**Measures:** test nRMSE when the 3 most important features (ranked by the v2 feature ranking) lose 5%..50% of their cells. **Defect:** MCAR; 9 features, 4 informative. **Conditions:** `clean`, `mean`, `median`, `iterative`, plus `native` reference.

In [ ]:
run_experiment("c07_mcar_most", "Completeness - MCAR in the most informative features", "Percentage of missing values", axis([round(0.05 * k, 2) for k in range(1, 11)]),
               make_data_9_4, pollute_most, models=MODELS_NATIVE, conditions=["clean", "mean", "median", "iterative", "native"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c07 = show("c07_mcar_most")

## Experiment 8. MCAR in the less informative features

**Measures:** as C7 for the 3 least important features. **Defect:** MCAR; 9 features, 4 informative. **Conditions:** `clean`, `mean`, `median`, `iterative`, plus `native` reference.

In [ ]:
run_experiment("c08_mcar_less", "Completeness - MCAR in the less informative features", "Percentage of missing values", axis([round(0.05 * k, 2) for k in range(1, 11)]),
               make_data_9_4, pollute_less, models=MODELS_NATIVE, conditions=["clean", "mean", "median", "iterative", "native"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c08 = show("c08_mcar_less")

## Experiment 9. MCAR mixing informative and non-informative features

**Measures:** test nRMSE when both the 3 least and the 3 most important features lose 5%..50% of their cells. **Defect:** MCAR; the 3 most and 3 least important features are both ranked once on the clean data (before any pollution, asserted disjoint), so the ranking is not recomputed on polluted data (the v1 title said MNAR, but the mechanism is MCAR); 9 features, 4 informative. **Conditions:** `clean`, `mean`, `median`, `iterative`, plus `native` reference.

In [ ]:
run_experiment("c09_mcar_mixed", "Completeness - MCAR mixing informative and non-informative features", "Percentage of missing values", axis([round(0.05 * k, 2) for k in range(1, 11)]),
               make_data_9_4, pollute_mixed, models=MODELS_NATIVE, conditions=["clean", "mean", "median", "iterative", "native"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c09 = show("c09_mcar_mixed")

## Experiment 10. MCAR on all features combined with MNAR on one feature

**Measures:** test nRMSE when all features have x (5%..50%) MCAR-missing cells and feature 0 additionally loses its upper-tail 25% (MNAR, `q_low=0.75, q_high=1.0`). The MNAR is applied first on the complete column and the MCAR afterwards, so the realised missing rate on feature 0 is the union, about 0.25 + 0.75 * x (e.g. about 0.29 at x = 5% and 0.63 at 50%); the other features have rate about x. The requested and realised per-column rates are in `meta`. **Defect:** MCAR + MNAR; 10 features, all informative. **Conditions:** `clean`, `mean`, `iterative`, `mean_indicator`, plus `native` reference.

In [ ]:
run_experiment("c10_mcar_mnar", "Completeness - MCAR on all features plus MNAR upper tail on one feature", "MCAR percentage of missing values", axis([round(0.05 * k, 2) for k in range(1, 11)]),
               make_data_10_10, pollute_mcar_mnar, models=MODELS_NATIVE, conditions=["clean", "mean", "iterative", "mean_indicator", "native"],
               seeds=SEEDS, n_jobs=N_JOBS)
summary_c10 = show("c10_mcar_mnar")

## v3 changes

Compared with `completeness_v2.ipynb` (method differences apply to every experiment unless stated):

- **Protocol (all):** the imputer, scaler and target scaler are fitted on the training fold only (v2 imputed the whole table before the split); 5 shuffled 70/30 folds per seed; metrics in target units and nRMSE = RMSE / std of the clean target. Data come from `make_regression_data` (signal plus Gaussian noise of 20% of the signal std) instead of noiseless `make_regression`, so the irreducible error is known.
- **Seeds and bands (all):** 3 seeds (0, 1, 2) vary data, pollution and split (v2 used the single seed 2023). Folds are averaged within a seed; the band in the plots is mean +- SD over the 3 seeds, a descriptive spread across repetitions and not a confidence interval. Conditions and models are paired (same data, mask and folds).
- **Reference (all):** a `clean` condition is reported in every experiment; `HGBRegressor` with `native` NaN handling is a separate reference in every polluted experiment (C1-C10), not an imputation method.
- **Models (all):** the six study models; GP lower length-scale bound 0.1 (v2 default bounds let the optimiser collapse to the mean predictor); bound hits and convergence problems are recorded in `status`.
- **C0:** now a learning curve from n = 100 to 1000 (v2: 1000..1090); data regenerated per n via `make_data(seed, x)`; `clean` only.
- **C1:** the v2 regression-imputation variant is replaced by `iterative` (BayesianRidge); `knn` scales features first; `mean_indicator` added; `std_constant` is labelled a naive constant (not a recognised method) and `most_frequent` degenerate on continuous data. **C2:** same set without `most_frequent` and `std_constant`, plus `native`.
- **C3:** x is the feature rank by true |coefficient| (columns ordered with `shuffle=False, coef=True`), so informative vs noise features are identifiable; the `informative` flag is in `meta` (v2 used shuffled columns, so index 0..9 had no meaning).
- **C4:** n from 100 to 1000 (v2: 1000..1090) with data regenerated per n; the v2 polluter seeds every column identically, so the missing-row sets are nested across columns (kept).
- **C5/C6:** x is now the missing rate (5%..50%) via `q = 1 - x` rather than the sample size with a fixed 20% rate; C6 is an upper-tail MNAR (v2 removed the central 25..75% band), a cleaner self-masking mechanism with a controllable rate. `mean_indicator` added.
- **C7/C8:** unchanged mechanism (3 most / least important features); `native` and `clean` references added.
- **C9:** renamed "MCAR mixing informative and non-informative features" because the mechanism is MCAR (v1 title said MNAR).
- **C9:** both feature sets are ranked on the clean data (v2 re-ranked on already polluted data), asserted disjoint.
- **C10:** the MNAR part is the upper tail of feature 0 at a fixed 25% of the complete column (applied before the MCAR; the union rate is about 0.25 + 0.75 * x) (v2 moved the MNAR feature and used the central band, which confounded feature and mechanism).